# Практика 2 — вариант 24
Соколовский Владислав, Ист-05/26б. Базовое извлечение.

In [1]:
import os
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd
load_dotenv()
assert os.getenv("GIGA_KEY"), "Создайте локальный .env с GIGA_KEY"
llm = GigaChat(credentials=os.environ["GIGA_KEY"], model="GigaChat-2", temperature=0.2, max_tokens=1000, verify_ssl_certs=False)
print(llm.invoke("Привет!").content)

Привет!


In [2]:
prompt = ChatPromptTemplate.from_messages([("system", "Ты эксперт по заявкам на аренду жилья."), ("human", "Заявка: {text}\nОпредели количество проживающих, включая детей. Верни только целое число.")])
chain = prompt | llm | StrOutputParser()
df = pd.read_csv("rental_24.csv", sep=";")
assert len(df) == 15
results = []
for t in df.text:
    try:
        results.append(chain.invoke({"text": t}))
    except Exception as e:
        results.append("ERROR: " + type(e).__name__)
df["result"] = results
display(df)

0  2  2
1  4  4
2  5  5
3  4  4
4  3  3
5  3  Количество проживающих в номере не указано.
6  6  7
7  2  2
8  2  2
9  3  3
10  4  Всего проживает 4 человека.
11  4  4
12  1  Один человек плюс две собаки – всего **3** проживающих.
13  2  2
14  3  3

In [3]:
df["correct"] = pd.to_numeric(df["result"], errors="coerce").eq(df["amount"])
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
if df.result.str.startswith("ERROR:").all():
    print("API недоступен: точность не измерена")
else:
    print(f"Верно: {df.correct.sum()}; ошибок: {(~df.correct).sum()}; точность: {df.correct.mean():.1%}")
display(df.loc[~df.correct])

Верно: 11; ошибок: 4; точность: 73.3%


5  3  Количество проживающих в номере не указано.  False
6  6  7  False
10  4  Всего проживает 4 человека.  False
12  1  Один человек плюс две собаки – всего **3** проживающих.  False